In [ ]:
# DuckDuckGo, 트래픽 급증 속 ‘No-AI’ 검색 엔진에 더 쉽게 접근할 수 있게 함
# 실시간 웹 검색 구현
!pip install -U langchain-openai langchain-community duckduckgo-search ddgs


In [ ]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.prompts import ChatPromptTemplate
import time
import os
from dotenv import load_dotenv

load_dotenv()

class MyRealtimeWebRag:
    def __init__(self):
        self.search = DuckDuckGoSearchResults()

        self.llm = ChatOpenAI(
            model = "gpt-4o-mini",
            temperature=0.0,
            api_key=os.getenv("OPENAI_API_KEY")
        )
        message = """
            웹에서 검색한 최신정보를 바탕으로 답변하세요.

            검색 결과 :
            {search_results}

            질문:
            {question}

            중요:
            검색 결과에 있는 답변만 하세요. 추측하지 마세요.
            모르면 모른다고 하세요.

            답변:
        """

        self.qa_prompt = ChatPromptTemplate.from_messages(
            [
                ("human", message)    # 'role':'user'와 유사,
            ]
        )

    def answerFunc(self, question):
        print(f'검색 중 : {question}')

        search_results = self.search.run(question)  # DuckDuckGo로 질문 관련 웹 검색 실행
        time.sleep(1)
        qa_chain = self.qa_prompt | self.llm   # prompt와 llm을 체인으로 연결

        response = qa_chain.invoke(  # LLM에게 prompt에 대한 답변 요청
            {
                "search_results":search_results,    # 검색결과를 prompt의 {search_results}에 전달
                "question":question   # 사용자 질문을 prompt의 {question}에 전달
            }
        )

        return response

if __name__ == '__main__':
    web_rag = MyRealtimeWebRag()

    questions = [
        "최신 AI 기술은 뭐니?",
        "한국에서 가장 인기 있는 빵은?",
        "한국의 여름철 장마에 대해 알려줘"
    ]

    for q in questions:
        print(f'\n질문:{q}')
        answer = web_rag.answerFunc(q)
        print(f'답변:{answer.content}')
        print()
